In [0]:
# Databricks notebook source

# =============================================================================
# NSSC TABLE 1 - NON-GOVERNMENT SCHOOLS
# Template-aligned standalone Python / PySpark version
#
# Student source:
#   pr_dap_efs.efs_gold.ngc_student_cohort_extract
#
# School/geography source:
#   pr_dap_efs.business_sdpt_nongov_census.schools_geography_2026
#
# Source filter:
#   CalendarYear = 26
#
# Output year:
#   2026
#
# Final Delta table:
#   pr_dap_efs.business_sdpt_abs.nssc_ngs_table_1_2026
#
# SPECIAL SCHOOL RULE:
#   Both "Special" and "Special Assistance" school types are
#   categorised as Special (schooltype = 4, special_ind = 'Special').
#
# AFFILIATION RULE:
#   Resolved once from the schools geography table via a dedicated
#   affiliation lookup (school_location_affiliation_base).
#   Priority-ordered checks prevent "non-anglican" matching as
#   Anglican and "non-catholic" matching as Catholic.
#   Template codes:  2 = Anglican,  3 = Catholic,  4 = Other.
# =============================================================================

from functools import reduce
import re

from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType


# =============================================================================
# 1. CONFIGURATION
# =============================================================================

CALENDAR_YEAR = 26
CENSUS_YEAR = 2026

STUDENT_SOURCE_TABLE = (
    "pr_dap_efs.efs_gold.ngc_student_cohort_extract"
)

SCHOOL_SOURCE_TABLE = (
    "pr_dap_efs.business_sdpt_nongov_census."
    f"schools_geography_{CENSUS_YEAR}"
)

OUTPUT_TABLE = (
    "pr_dap_efs.business_sdpt_abs."
    f"nssc_ngs_table_1_{CENSUS_YEAR}"
)

DECIMAL_TYPE = DecimalType(24, 6)


# =============================================================================
# 2. HELPER FUNCTIONS
# =============================================================================

def find_column(dataframe, candidates, required=True):
    """
    Return the first available column from a list of candidates.
    Matching is case-insensitive.
    """

    available = {
        name.lower(): name
        for name in dataframe.columns
    }

    for candidate in candidates:

        if candidate.lower() in available:

            return available[
                candidate.lower()
            ]

    if required:

        raise ValueError(
            "None of the expected columns were found.\n"
            f"Expected one of: {', '.join(candidates)}\n"
            f"Available columns: {', '.join(dataframe.columns)}"
        )

    return None


def quoted_name(column_name):
    """
    Escape a physical Spark column name.
    """

    return column_name.replace(
        "`",
        "``"
    )


def safe_col(column_name):
    """
    Safely reference a physical Spark column, including columns
    containing literal dots.
    """

    return F.col(
        f"`{quoted_name(column_name)}`"
    )


def cleaned_text(column_name):
    """
    Convert a physical source field to trimmed text.
    """

    return F.trim(
        safe_col(column_name).cast("string")
    )


def try_integer(column_name):
    """
    Safely convert a physical source field to INT.

    Blank and malformed values return NULL.
    """

    escaped = quoted_name(column_name)

    return F.expr(
        f"""
        try_cast(
            NULLIF(
                TRIM(
                    CAST(`{escaped}` AS STRING)
                ),
                ''
            )
            AS INT
        )
        """
    )


def try_decimal(column_name):
    """
    Safely convert a physical source field to DECIMAL(24,6).

    Blank, malformed and NULL values become zero.
    """

    escaped = quoted_name(column_name)

    return F.coalesce(
        F.expr(
            f"""
            try_cast(
                NULLIF(
                    TRIM(
                        CAST(`{escaped}` AS STRING)
                    ),
                    ''
                )
                AS DECIMAL(24,6)
            )
            """
        ),
        F.lit(0).cast("decimal(24,6)")
    )


def size_band(column_name):
    """
    Convert Primary or Secondary FTE to the template size code.

    1  = 0
    2  = 1-20
    3  = 21-35
    4  = 36-100
    ...
    18 = 1401-1500
    19 = 1501 and over
    """

    value = F.coalesce(
        F.col(column_name),
        F.lit(0).cast(DECIMAL_TYPE)
    )

    return (
        F.when(value <= 0, 1)
        .when(value <= 20, 2)
        .when(value <= 35, 3)
        .when(value <= 100, 4)
        .when(value <= 200, 5)
        .when(value <= 300, 6)
        .when(value <= 400, 7)
        .when(value <= 500, 8)
        .when(value <= 600, 9)
        .when(value <= 700, 10)
        .when(value <= 800, 11)
        .when(value <= 900, 12)
        .when(value <= 1000, 13)
        .when(value <= 1100, 14)
        .when(value <= 1200, 15)
        .when(value <= 1300, 16)
        .when(value <= 1400, 17)
        .when(value <= 1500, 18)
        .otherwise(19)
    )


# =============================================================================
# 3. LOAD AND FILTER STUDENT SOURCE
# =============================================================================
# Deduplication occurs on the original source rows before columns are removed.

student_raw = spark.table(
    STUDENT_SOURCE_TABLE
)

calendar_year_column = find_column(
    student_raw,
    [
        "CalendarYear",
        "calendar_year"
    ]
)

student_source = (
    student_raw

    .filter(
        try_integer(
            calendar_year_column
        ) == CALENDAR_YEAR
    )

    .dropDuplicates()
)

student_source.createOrReplaceTempView(
    "ngc_student_cohort_source"
)


# =============================================================================
# 4. IDENTIFY STUDENT COLUMNS
# =============================================================================

student_school_ageid_column = find_column(
    student_source,
    [
        "school_ageid",
        "School_AGEID",
        "AGEID"
    ]
)

student_location_ageid_column = find_column(
    student_source,
    [
        "location_ageid",
        "School_Location_AGEID",
        "Location_AGEID"
    ]
)

student_school_name_column = find_column(
    student_source,
    [
        "school_name",
        "SchoolName",
        "School_Name"
    ],
    required=False
)

student_state_column = find_column(
    student_source,
    [
        "school_state",
        "School_State",
        "state",
        "State"
    ]
)

student_school_type_column = find_column(
    student_source,
    [
        "school_type",
        "SchoolType",
        "School_Type",
        "centre_type"
    ],
    required=False
)

student_school_level_column = find_column(
    student_source,
    [
        "school_education_level",
        "location_education_level",
        "school_level",
        "SchoolLevel"
    ],
    required=False
)

student_year_level_column = find_column(
    student_source,
    [
        "year_level",
        "YearLevel",
        "grade",
        "Grade"
    ]
)

student_headcount_column = find_column(
    student_source,
    [
        "headcount",
        "HeadCount",
        "enrolments",
        "student_count"
    ]
)

student_fte_column = find_column(
    student_source,
    [
        "fte",
        "FTE",
        "student_fte"
    ]
)


# =============================================================================
# 5. LOAD SCHOOL/GEOGRAPHY SOURCE
# =============================================================================

school_raw = spark.table(
    SCHOOL_SOURCE_TABLE
)


# =============================================================================
# 6. IDENTIFY SCHOOL/GEOGRAPHY COLUMNS
# =============================================================================

school_ageid_column = find_column(
    school_raw,
    [
        "AGEID",
        "school_ageid",
        "School_AGEID"
    ]
)

school_location_ageid_column = find_column(
    school_raw,
    [
        "School_Location_AGEID",
        "location_ageid",
        "Location_AGEID"
    ]
)

school_name_column = find_column(
    school_raw,
    [
        "SchoolName",
        "school_name",
        "School_Name"
    ],
    required=False
)

school_state_column = find_column(
    school_raw,
    [
        "School_State",
        "school_state",
        "state",
        "State"
    ],
    required=False
)

school_affiliation_column = find_column(
    school_raw,
    [
        "Affiliation",
        "affiliation",
        "school_affiliation",
        "School_Affiliation",
        
    ],
    required=False
)

school_type_column = find_column(
    school_raw,
    [
        "SchoolType",
        "school_type",
        "School_Type",
        "centre_type"
    ],
    required=False
)

school_level_column = find_column(
    school_raw,
    [
        "SchoolLevel",
        "school_level",
        "school_education_level",
        "location_education_level"
    ],
    required=False
)


# =============================================================================
# 7. STANDARDISE SCHOOL/GEOGRAPHY SOURCE (excluding affiliation)
# =============================================================================

school_standard = (
    school_raw

    .select(
        safe_col(
            school_ageid_column
        ).cast("string").alias(
            "geo_school_ageid"
        ),

        safe_col(
            school_location_ageid_column
        ).cast("string").alias(
            "geo_location_ageid"
        ),

        (
            cleaned_text(
                school_name_column
            )
            if school_name_column
            else F.lit(None).cast("string")
        ).alias(
            "geo_school_name"
        ),

        (
            cleaned_text(
                school_state_column
            )
            if school_state_column
            else F.lit(None).cast("string")
        ).alias(
            "geo_state"
        ),

        (
            cleaned_text(
                school_type_column
            )
            if school_type_column
            else F.lit(None).cast("string")
        ).alias(
            "geo_school_type"
        ),

        (
            cleaned_text(
                school_level_column
            )
            if school_level_column
            else F.lit(None).cast("string")
        ).alias(
            "geo_school_level"
        )
    )

    .filter(
        F.col("geo_school_ageid").isNotNull()
        &
        F.col("geo_location_ageid").isNotNull()
    )
)


# =============================================================================
# 7A. SCHOOL / LOCATION AFFILIATION LOOKUP
# =============================================================================
# Affiliation is resolved once from the schools geography table using
# priority-ordered checks.  This is the single authoritative source
# for affiliation — it is NOT taken from the student extract.
#
# Priority order matters:
#   "non-anglican" BEFORE "anglican"   (both contain "anglican")
#   "non-catholic"  BEFORE "catholic"  (both contain "catholic")
#
# Template codes:
#   2 = Anglican
#   3 = Catholic
#   4 = Other  (non-Anglican, non-Catholic, everything else)
# =============================================================================

school_location_affiliation_base = (
    school_raw

    .filter(
        safe_col(
            school_ageid_column
        ).isNotNull()
        &
        safe_col(
            school_location_ageid_column
        ).isNotNull()
    )

    .select(
        safe_col(
            school_ageid_column
        ).cast("string").alias(
            "affil_school_ageid"
        ),

        safe_col(
            school_location_ageid_column
        ).cast("string").alias(
            "affil_location_ageid"
        ),

        F.lower(
            F.trim(
                safe_col(
                    school_affiliation_column
                ).cast("string")
            )
        ).alias(
            "affiliation_text"
        )
    )

    .withColumn(
        "school_affiliation",

        F.when(
            F.col("affiliation_text").contains(
                "non-anglican"
            ),
            4
        )

        .when(
            F.col("affiliation_text").contains(
                "anglican"
            ),
            2
        )

        .when(
            F.col("affiliation_text").contains(
                "non-catholic"
            ),
            4
        )

        .when(
            F.col("affiliation_text").contains(
                "catholic"
            ),
            3
        )

        .otherwise(
            4
        )
    )

    .select(
        "affil_school_ageid",
        "affil_location_ageid",
        "school_affiliation"
    )

    .dropDuplicates()
)

school_location_affiliation_base.createOrReplaceTempView(
    "school_location_affiliation_base"
)


# =============================================================================
# 8. CHECK SCHOOL/LOCATION CONFLICTS
# =============================================================================
# A school/location combination should not have conflicting classifications.

school_location_conflicts = (
    school_standard

    .groupBy(
        "geo_school_ageid",
        "geo_location_ageid"
    )

    .agg(
        F.countDistinct(
            "geo_state"
        ).alias(
            "state_count"
        ),

        F.countDistinct(
            "geo_school_type"
        ).alias(
            "school_type_count"
        ),

        F.countDistinct(
            "geo_school_level"
        ).alias(
            "school_level_count"
        )
    )

    .filter(
        (F.col("state_count") > 1)
        |
        (F.col("school_type_count") > 1)
        |
        (F.col("school_level_count") > 1)
    )
)

school_location_conflict_count = (
    school_location_conflicts.count()
)

if school_location_conflict_count > 0:

    display(
        school_location_conflicts
    )

    raise ValueError(
        f"{school_location_conflict_count} school/location "
        "combinations contain conflicting classifications."
    )


# =============================================================================
# 8A. CHECK AFFILIATION CONFLICTS
# =============================================================================
# Each school/location should resolve to exactly one affiliation code.

affiliation_conflicts = (
    school_location_affiliation_base

    .groupBy(
        "affil_school_ageid",
        "affil_location_ageid"
    )

    .agg(
        F.countDistinct(
            "school_affiliation"
        ).alias(
            "affiliation_count"
        )
    )

    .filter(
        F.col("affiliation_count") > 1
    )
)

affiliation_conflict_count = (
    affiliation_conflicts.count()
)

if affiliation_conflict_count > 0:

    display(
        affiliation_conflicts
    )

    raise ValueError(
        f"{affiliation_conflict_count} school/location "
        "combinations have conflicting affiliation codes."
    )


# =============================================================================
# 9. CREATE UNIQUE SCHOOL/LOCATION MAPPING (excluding affiliation)
# =============================================================================

school_unique = (
    school_standard

    .groupBy(
        "geo_school_ageid",
        "geo_location_ageid"
    )

    .agg(
        F.first(
            "geo_school_name",
            ignorenulls=True
        ).alias(
            "geo_school_name"
        ),

        F.first(
            "geo_state",
            ignorenulls=True
        ).alias(
            "geo_state"
        ),

        F.first(
            "geo_school_type",
            ignorenulls=True
        ).alias(
            "geo_school_type"
        ),

        F.first(
            "geo_school_level",
            ignorenulls=True
        ).alias(
            "geo_school_level"
        )
    )
)


# =============================================================================
# 9A. CREATE UNIQUE AFFILIATION MAPPING
# =============================================================================

affiliation_unique = (
    school_location_affiliation_base

    .groupBy(
        "affil_school_ageid",
        "affil_location_ageid"
    )

    .agg(
        F.first(
            "school_affiliation",
            ignorenulls=True
        ).alias(
            "school_affiliation"
        )
    )
)


# =============================================================================
# 10. STANDARDISE STUDENT SOURCE
# =============================================================================
# Note: student affiliation column is NOT selected — affiliation comes
# exclusively from the schools geography table.

student_standard = (
    student_source

    .select(
        safe_col(
            student_school_ageid_column
        ).cast("string").alias(
            "school_ageid"
        ),

        safe_col(
            student_location_ageid_column
        ).cast("string").alias(
            "location_ageid"
        ),

        (
            cleaned_text(
                student_school_name_column
            )
            if student_school_name_column
            else F.lit(None).cast("string")
        ).alias(
            "student_school_name"
        ),

        cleaned_text(
            student_state_column
        ).alias(
            "student_state"
        ),

        (
            cleaned_text(
                student_school_type_column
            )
            if student_school_type_column
            else F.lit(None).cast("string")
        ).alias(
            "student_school_type"
        ),

        (
            cleaned_text(
                student_school_level_column
            )
            if student_school_level_column
            else F.lit(None).cast("string")
        ).alias(
            "student_school_level"
        ),

        cleaned_text(
            student_year_level_column
        ).alias(
            "year_level"
        ),

        try_decimal(
            student_headcount_column
        ).alias(
            "headcount"
        ),

        try_decimal(
            student_fte_column
        ).alias(
            "fte_rate"
        )
    )

    .withColumn(
        "student_fte",

        F.col("headcount")
        * F.col("fte_rate")
    )
)


# =============================================================================
# 11. JOIN STUDENTS TO SCHOOL/LOCATION AND AFFILIATION SOURCES
# =============================================================================
# Geography attributes come from school_unique.
# Affiliation comes exclusively from affiliation_unique.

student_joined = (
    student_standard.alias(
        "student"
    )

    .join(
        school_unique.alias(
            "school"
        ),

        (
            F.col("student.school_ageid")
            == F.col("school.geo_school_ageid")
        )
        &
        (
            F.col("student.location_ageid")
            == F.col("school.geo_location_ageid")
        ),

        "left"
    )

    .join(
        affiliation_unique.alias(
            "affil"
        ),

        (
            F.col("student.school_ageid")
            == F.col("affil.affil_school_ageid")
        )
        &
        (
            F.col("student.location_ageid")
            == F.col("affil.affil_location_ageid")
        ),

        "left"
    )

    .select(
        "student.*",

        F.coalesce(
            F.col("student.student_school_name"),
            F.col("school.geo_school_name")
        ).alias(
            "school_name"
        ),

        F.coalesce(
            F.col("student.student_state"),
            F.col("school.geo_state")
        ).alias(
            "state_source"
        ),

        # Affiliation from the dedicated affiliation lookup only
        F.col(
            "affil.school_affiliation"
        ).alias(
            "affiliation"
        ),

        F.coalesce(
            F.col("student.student_school_type"),
            F.col("school.geo_school_type")
        ).alias(
            "school_type_source"
        ),

        F.coalesce(
            F.col("student.student_school_level"),
            F.col("school.geo_school_level")
        ).alias(
            "school_level_source"
        )
    )
)


# =============================================================================
# 12. MAP STATE TO TEMPLATE CODE
# =============================================================================
# Template:
#   1 NSW
#   2 Victoria
#   3 Queensland
#   4 South Australia
#   5 Western Australia
#   6 Tasmania
#   7 Northern Territory
#   8 Australian Capital Territory

student_joined = (
    student_joined

    .withColumn(
        "state_text",

        F.lower(
            F.trim(
                F.col("state_source")
            )
        )
    )

    .withColumn(
        "state",

        F.when(
            F.col("state_text").isin(
                "new south wales",
                "nsw",
                "1"
            ),
            1
        )

        .when(
            F.col("state_text").isin(
                "victoria",
                "vic",
                "2"
            ),
            2
        )

        .when(
            F.col("state_text").isin(
                "queensland",
                "qld",
                "3"
            ),
            3
        )

        .when(
            F.col("state_text").isin(
                "south australia",
                "sa",
                "4"
            ),
            4
        )

        .when(
            F.col("state_text").isin(
                "western australia",
                "wa",
                "5"
            ),
            5
        )

        .when(
            F.col("state_text").isin(
                "tasmania",
                "tas",
                "6"
            ),
            6
        )

        .when(
            F.col("state_text").isin(
                "northern territory",
                "nt",
                "7"
            ),
            7
        )

        .when(
            F.col("state_text").isin(
                "australian capital territory",
                "act",
                "8"
            ),
            8
        )
    )
)


# =============================================================================
# 13. (AFFILIATION ALREADY RESOLVED)
# =============================================================================
# Affiliation was mapped in Step 7A and joined in Step 11.
# The "affiliation" column already contains the template code
# (2 = Anglican, 3 = Catholic, 4 = Other).
# No further mapping is required.


# =============================================================================
# 14. MAP YEAR LEVEL TO TEMPLATE GRADE CODE
# =============================================================================
# Template:
#   1  Pre-grade 1
#   2  Grade 1
#   ...
#   13 Grade 12

student_joined = (
    student_joined

    .withColumn(
        "year_level_text",

        F.lower(
            F.trim(
                F.col("year_level")
            )
        )
    )

    .withColumn(
        "year_number_text",

        F.regexp_extract(
            F.col("year_level_text"),
            r"(\d+)",
            1
        )
    )

    .withColumn(
        "year_number",

        F.expr(
            """
            try_cast(
                NULLIF(
                    year_number_text,
                    ''
                )
                AS INT
            )
            """
        )
    )

    .withColumn(
        "grade_code",

        F.when(
            F.col("year_level_text").isin(
                "foundation",
                "prep",
                "pre-primary",
                "pre primary",
                "kindergarten",
                "reception",
                "transition",
                "pre-grade 1",
                "pre grade 1",
                "11 years and younger"
            ),
            1
        )

        .when(
            F.col("year_number").between(
                1,
                12
            ),
            F.col("year_number") + 1
        )

        .when(
            F.col("year_level_text")
            == "12 years and older",
            13
        )
    )
)


# =============================================================================
# 15. CLASSIFY SPECIAL SCHOOL STATUS
# =============================================================================
# Both "Special" and "Special Assistance" school types are classified
# as Special (schooltype = 4, special_ind = 'Special').
#
# contains("special") covers both because "special assistance"
# contains the substring "special".

student_joined = (
    student_joined
    .withColumn(
        "school_type_text",
        F.lower(
            F.trim(
                F.regexp_replace(
                    F.col("school_type_source"),
                    r"\s+",
                    " "
                )
            )
        )
    )
    .withColumn(
        "is_special",
        F.when(
            F.col("school_type_text").isin(
                "special",
                "special school",
                "special assistance",
                "special assistance school"
            ),
            1
        ).otherwise(0)))


# =============================================================================
# 16. VALIDATE GEOGRAPHY JOIN TOTALS
# =============================================================================

before_join_totals = (
    student_standard

    .agg(
        F.sum(
            "headcount"
        ).alias(
            "before_headcount"
        ),

        F.sum(
            "student_fte"
        ).alias(
            "before_fte"
        )
    )
)

after_join_totals = (
    student_joined

    .agg(
        F.sum(
            "headcount"
        ).alias(
            "after_headcount"
        ),

        F.sum(
            "student_fte"
        ).alias(
            "after_fte"
        )
    )
)

join_validation = (
    before_join_totals

    .crossJoin(
        after_join_totals
    )

    .withColumn(
        "headcount_difference",

        F.col("after_headcount")
        - F.col("before_headcount")
    )

    .withColumn(
        "fte_difference",

        F.col("after_fte")
        - F.col("before_fte")
    )

    .withColumn(
        "status",

        F.when(
            (
                F.abs(
                    F.col("headcount_difference")
                ) < F.lit(0.0001)
            )
            &
            (
                F.abs(
                    F.col("fte_difference")
                ) < F.lit(0.0001)
            ),
            "MATCH"
        )

        .otherwise(
            "CHECK"
        )
    )
)

if join_validation.first()["status"] != "MATCH":

    display(
        join_validation
    )

    raise ValueError(
        "The school/location join changed the student "
        "headcount or FTE totals."
    )


# =============================================================================
# 17. VALIDATE SCHOOL-LEVEL STATE AND AFFILIATION CONSISTENCY
# =============================================================================

school_attribute_conflicts = (
    student_joined

    .groupBy(
        "school_ageid"
    )

    .agg(
        F.countDistinct(
            "state"
        ).alias(
            "state_count"
        ),

        F.countDistinct(
            "affiliation"
        ).alias(
            "affiliation_count"
        )
    )

    .filter(
        (F.col("state_count") > 1)
        |
        (F.col("affiliation_count") > 1)
    )
)

school_attribute_conflict_count = (
    school_attribute_conflicts.count()
)

if school_attribute_conflict_count > 0:

    display(
        school_attribute_conflicts
    )

    raise ValueError(
        f"{school_attribute_conflict_count} schools have "
        "conflicting state or affiliation classifications."
    )


# =============================================================================
# 18. CREATE SCHOOL-LEVEL ATTRIBUTES
# =============================================================================

school_attributes = (
    student_joined

    .groupBy(
        "school_ageid"
    )

    .agg(
        F.first(
            "state",
            ignorenulls=True
        ).alias(
            "state"
        ),

        F.first(
            "affiliation",
            ignorenulls=True
        ).alias(
            "affiliation"
        ),

        F.max(
            "is_special"
        ).alias(
            "special_flag"
        ),

        F.first(
            "school_name",
            ignorenulls=True
        ).alias(
            "school_name"
        )
    )
)


# =============================================================================
# 19. CREATE SCHOOL GRADE RANGE
# =============================================================================

school_grades = (
    student_joined

    .groupBy(
        "school_ageid"
    )

    .agg(
        F.min(
            "grade_code"
        ).alias(
            "lowgrade"
        ),

        F.max(
            "grade_code"
        ).alias(
            "highgrade"
        )
    )
)


# =============================================================================
# 20. CREATE SCHOOL TYPE AND SPECIAL INDICATOR
# =============================================================================
# Template:
#   1 Primary
#   2 Secondary
#   3 Combined
#   4 Special

school_classification = (
    school_attributes.alias(
        "attributes"
    )

    .join(
        school_grades.alias(
            "grades"
        ),

        "school_ageid",

        "left"
    )

    .withColumn(
        "schooltype",

        F.when(
            F.col("special_flag") == 1,
            4
        )

        .when(
            F.col("lowgrade").between(
                1,
                7
            )
            &
            F.col("highgrade").between(
                1,
                7
            ),
            1
        )

        .when(
            F.col("lowgrade").between(
                8,
                13
            )
            &
            F.col("highgrade").between(
                8,
                13
            ),
            2
        )

        .when(
            F.col("lowgrade").between(
                1,
                7
            )
            &
            F.col("highgrade").between(
                8,
                13
            ),
            3
        )
    )

   .withColumn(
    "special_ind",
    F.when(
        F.col("schooltype") == 4,
        "Special"
    )
    .otherwise(
        "Regular"
    )
)

    .select(
        "school_ageid",
        "school_name",
        "state",
        "affiliation",
        "lowgrade",
        "highgrade",
        "schooltype",
        "special_ind"
    )
)


# =============================================================================
# 21. CALCULATE PRIMARY AND SECONDARY FTE
# =============================================================================
# School-size bands must use FTE, not headcount.

school_phase_fte = (
    student_joined

    .groupBy(
        "school_ageid"
    )

    .agg(
        F.sum(
            F.when(
                F.col("grade_code").between(
                    1,
                    7
                ),
                F.col("student_fte")
            )

            .otherwise(
                F.lit(0)
            )
        ).alias(
            "primary_fte"
        ),

        F.sum(
            F.when(
                F.col("grade_code").between(
                    8,
                    13
                ),
                F.col("student_fte")
            )

            .otherwise(
                F.lit(0)
            )
        ).alias(
            "secondary_fte"
        )
    )

    .withColumn(
        "primary_size",

        size_band(
            "primary_fte"
        )
    )

    .withColumn(
        "secondary_size",

        size_band(
            "secondary_fte"
        )
    )
)


# =============================================================================
# 22. CREATE SCHOOL-LEVEL ENROLMENTS AND FTE
# =============================================================================

school_measures = (
    student_joined

    .groupBy(
        "school_ageid"
    )

    .agg(
        F.sum(
            "headcount"
        ).alias(
            "enrolments"
        ),

        F.sum(
            "student_fte"
        ).alias(
            "fte"
        )
    )
)


# =============================================================================
# 23. CREATE ONE ROW PER SCHOOL
# =============================================================================

nssc_school_dimensions = (
    school_measures.alias(
        "measures"
    )

    .join(
        school_classification.alias(
            "classification"
        ),

        "school_ageid",

        "left"
    )

    .join(
        school_phase_fte.alias(
            "size"
        ),

        "school_ageid",

        "left"
    )

    .select(
        F.lit(
            CENSUS_YEAR
        ).cast("int").alias(
            "year"
        ),

        F.col(
            "classification.state"
        ).cast("int").alias(
            "state"
        ),

        F.col(
            "classification.affiliation"
        ).cast("int").alias(
            "affiliation"
        ),

        F.col(
            "classification.lowgrade"
        ).cast("int").alias(
            "lowgrade"
        ),

        F.col(
            "classification.highgrade"
        ).cast("int").alias(
            "highgrade"
        ),

        F.col(
            "classification.schooltype"
        ).cast("int").alias(
            "schooltype"
        ),

        F.col(
            "classification.special_ind"
        ).cast("string").alias(
            "special_ind"
        ),

        F.col(
            "size.primary_size"
        ).cast("int").alias(
            "primary_size"
        ),

        F.col(
            "size.secondary_size"
        ).cast("int").alias(
            "secondary_size"
        ),

        F.col(
            "measures.enrolments"
        ).cast(
            DECIMAL_TYPE
        ).alias(
            "enrolments"
        ),

        F.col(
            "measures.fte"
        ).cast(
            DECIMAL_TYPE
        ).alias(
            "fte"
        ),

        F.col(
            "classification.school_name"
        ).alias(
            "school_name"
        ),

        F.col(
            "school_ageid"
        )
    )
)

nssc_school_dimensions.createOrReplaceTempView(
    "nssc_school_dimensions"
)


# =============================================================================
# 24. TEMPLATE VALUE VALIDATION
# =============================================================================

template_validation_issues = (
    nssc_school_dimensions

    .filter(
        (F.col("year") != CENSUS_YEAR)

        | F.col("state").isNull()
        | ~F.col("state").between(1, 8)

        | F.col("affiliation").isNull()
        | ~F.col("affiliation").isin(2, 3, 4)

        | F.col("lowgrade").isNull()
        | ~F.col("lowgrade").between(1, 13)

        | F.col("highgrade").isNull()
        | ~F.col("highgrade").between(1, 13)

        | (F.col("lowgrade") > F.col("highgrade"))

        | F.col("schooltype").isNull()
        | ~F.col("schooltype").isin(1, 2, 3, 4)

        | F.col("special_ind").isNull()
        | ~F.col("special_ind").isin(
            "Regular",
            "Special"
        )

        | F.col("primary_size").isNull()
        | ~F.col("primary_size").between(1, 19)

        | F.col("secondary_size").isNull()
        | ~F.col("secondary_size").between(1, 19)

        | F.col("enrolments").isNull()
        | (F.col("enrolments") < 0)

        | F.col("fte").isNull()
        | (F.col("fte") < 0)
    )

    .orderBy(
        "state",
        "school_ageid"
    )
)


# =============================================================================
# 25. CROSS-FIELD CONSISTENCY VALIDATION
# =============================================================================

schooltype_consistency_issues = (
    nssc_school_dimensions

    .filter(
        (
            (F.col("schooltype") == 1)
            &
            ~(
                F.col("lowgrade").between(1, 7)
                &
                F.col("highgrade").between(1, 7)
            )
        )

        |

        (
            (F.col("schooltype") == 2)
            &
            ~(
                F.col("lowgrade").between(8, 13)
                &
                F.col("highgrade").between(8, 13)
            )
        )

        |

        (
            (F.col("schooltype") == 3)
            &
            ~(
                F.col("lowgrade").between(1, 7)
                &
                F.col("highgrade").between(8, 13)
            )
        )

        |

        (
            (F.col("schooltype") == 4)
            &
            (
                F.col("special_ind") != "Special"
            )
        )

        |

        (
            F.col("schooltype").isin(
                1,
                2,
                3
            )
            &
            (
                F.col("special_ind") != "Regular"
            )
        )
    )

    .orderBy(
        "state",
        "school_ageid"
    )
)


# =============================================================================
# 26. DUPLICATE SCHOOL VALIDATION
# =============================================================================

duplicate_school_dimensions = (
    nssc_school_dimensions

    .groupBy(
        "year",
        "school_ageid"
    )

    .count()

    .filter(
        F.col("count") > 1
    )
)


# =============================================================================
# 27. UNRECOGNISED YEAR-LEVEL VALUES
# =============================================================================

unrecognised_year_levels = (
    student_joined

    .filter(
        F.col("grade_code").isNull()
    )

    .groupBy(
        "school_ageid",
        "school_name",
        "year_level"
    )

    .agg(
        F.sum(
            "headcount"
        ).alias(
            "headcount"
        ),

        F.sum(
            "student_fte"
        ).alias(
            "fte"
        )
    )

    .orderBy(
        "school_ageid",
        "year_level"
    )
)


# =============================================================================
# 28. STOP BEFORE OUTPUT IF TEMPLATE VALIDATION FAILS
# =============================================================================

template_issue_count = (
    template_validation_issues.count()
)

consistency_issue_count = (
    schooltype_consistency_issues.count()
)

duplicate_school_count = (
    duplicate_school_dimensions.count()
)

if duplicate_school_count > 0:

    print(
        f"{duplicate_school_count} schools have more than "
        "one school-level record."
    )

    display(
        duplicate_school_dimensions
    )


if template_issue_count > 0:

    print(
        f"{template_issue_count} schools have invalid or "
        "unresolved template values."
    )

    display(
        template_validation_issues
    )


if consistency_issue_count > 0:

    print(
        f"{consistency_issue_count} schools have inconsistent "
        "grade, school type or Special indicator values."
    )

    display(
        schooltype_consistency_issues
    )


if (
    duplicate_school_count > 0
    or template_issue_count > 0
    or consistency_issue_count > 0
):

    print(
        "Unrecognised source year levels:"
    )

    display(
        unrecognised_year_levels
    )

    raise ValueError(
        "NSSC Table 1 contains unresolved template inconsistencies. "
        "The final Delta table was not written."
    )


# =============================================================================
# 29. CREATE FINAL TABLE 1
# =============================================================================
# Column order matches the template:
#
# year, state, affiliation, lowgrade, highgrade, schooltype,
# special_ind, primary_size, secondary_size, school_count,
# enrolments, fte

nssc_ngs_table_1 = (
    nssc_school_dimensions

    .groupBy(
        "year",
        "state",
        "affiliation",
        "lowgrade",
        "highgrade",
        "schooltype",
        "special_ind",
        "primary_size",
        "secondary_size"
    )

    .agg(
        F.countDistinct(
            "school_ageid"
        ).cast("long").alias(
            "school_count"
        ),

        F.round(
            F.sum(
                "enrolments"
            ),
            0
        ).cast("long").alias(
            "enrolments"
        ),

        F.round(
            F.sum(
                "fte"
            ),
            2
        ).cast(
            "decimal(24,2)"
        ).alias(
            "fte"
        )
    )

    .select(
        F.col("year").cast("int").alias("year"),
        F.col("state").cast("int").alias("state"),
        F.col("affiliation").cast("int").alias("affiliation"),
        F.col("lowgrade").cast("int").alias("lowgrade"),
        F.col("highgrade").cast("int").alias("highgrade"),
        F.col("schooltype").cast("int").alias("schooltype"),
        F.col("special_ind").cast("string").alias("special_ind"),
        F.col("primary_size").cast("int").alias("primary_size"),
        F.col("secondary_size").cast("int").alias("secondary_size"),
        F.col("school_count").cast("long").alias("school_count"),
        F.col("enrolments").cast("long").alias("enrolments"),
        F.col("fte").cast("decimal(24,2)").alias("fte")
    )

    .orderBy(
        "year",
        "state",
        "affiliation",
        "schooltype",
        "lowgrade",
        "highgrade",
        "primary_size",
        "secondary_size"
    )
)

nssc_ngs_table_1.createOrReplaceTempView(
    "nssc_ngs_table_1"
)


# =============================================================================
# 30. RECONCILE SCHOOL COUNT, ENROLMENTS AND FTE
# =============================================================================

source_totals = (
    nssc_school_dimensions

    .agg(
        F.countDistinct(
            "school_ageid"
        ).alias(
            "source_school_count"
        ),

        F.sum(
            "enrolments"
        ).alias(
            "source_enrolments"
        ),

        F.sum(
            "fte"
        ).alias(
            "source_fte"
        )
    )
)

output_totals = (
    nssc_ngs_table_1

    .agg(
        F.sum(
            "school_count"
        ).alias(
            "output_school_count"
        ),

        F.sum(
            "enrolments"
        ).alias(
            "output_enrolments"
        ),

        F.sum(
            "fte"
        ).alias(
            "output_fte"
        )
    )
)

validation_df = (
    source_totals

    .crossJoin(
        output_totals
    )

    .withColumn(
        "school_count_difference",

        F.col("output_school_count")
        - F.col("source_school_count")
    )

    .withColumn(
        "enrolment_difference",

        F.col("output_enrolments")
        - F.col("source_enrolments")
    )

    .withColumn(
        "fte_difference",

        F.col("output_fte")
        - F.col("source_fte")
    )

    .withColumn(
        "status",

        F.when(
            (
                F.col("school_count_difference") == 0
            )
            &
            (
                F.abs(
                    F.col("enrolment_difference")
                ) < F.lit(0.01)
            )
            &
            (
                F.abs(
                    F.col("fte_difference")
                ) < F.lit(0.01)
            ),
            "MATCH"
        )

        .otherwise(
            "CHECK"
        )
    )
)


if validation_df.first()["status"] != "MATCH":

    display(
        validation_df
    )

    raise ValueError(
        "The final NSSC Table 1 does not reconcile with "
        "the school-level source dataset."
    )


# =============================================================================
# 31. CHECK DELTA-SAFE COLUMN NAMES
# =============================================================================

invalid_delta_columns = [
    column_name
    for column_name in nssc_ngs_table_1.columns
    if re.search(
        r'[ ,;{}()\n\t=]',
        column_name
    )
]

if invalid_delta_columns:

    raise ValueError(
        "The following output columns are not Delta-safe: "
        + ", ".join(
            invalid_delta_columns
        )
    )


# =============================================================================
# 32. SAVE FINAL DELTA TABLE
# =============================================================================

(
    nssc_ngs_table_1

    .write

    .format(
        "delta"
    )

    .mode(
        "overwrite"
    )

    .option(
        "overwriteSchema",
        "true"
    )

    .saveAsTable(
        OUTPUT_TABLE
    )
)


# =============================================================================
# 33. DISPLAY FINAL OUTPUT
# =============================================================================

print(
    f"Student source: {STUDENT_SOURCE_TABLE}"
)

print(
    f"School source: {SCHOOL_SOURCE_TABLE}"
)

print(
    f"CalendarYear filter: {CALENDAR_YEAR}"
)

print(
    f"NSSC reporting year: {CENSUS_YEAR}"
)

print(
    f"Final Delta table: {OUTPUT_TABLE}"
)

display(
    spark.table(
        OUTPUT_TABLE
    )
)


# =============================================================================
# 34. DISPLAY VALIDATION
# =============================================================================

print(
    "School count, enrolment and FTE reconciliation:"
)

display(
    validation_df
)


# =============================================================================
# 35. DISPLAY NON-BLOCKING YEAR-LEVEL DIAGNOSTICS
# =============================================================================

unrecognised_year_level_count = (
    unrecognised_year_levels.count()
)

if unrecognised_year_level_count > 0:

    print(
        f"Note: {unrecognised_year_level_count} school/year-level "
        "combinations were not mapped to template grade codes."
    )

    display(
        unrecognised_year_levels
    )

else:

    print(
        "Year-level validation passed: all source year levels "
        "mapped to template grade codes."
    )


print(
    "Completed successfully: NSSC Table 1 matches the template "
    "schema and all school count, enrolment and FTE totals reconcile."
)

Student source: pr_dap_efs.efs_gold.ngc_student_cohort_extract
School source: pr_dap_efs.business_sdpt_nongov_census.schools_geography_2026
CalendarYear filter: 26
NSSC reporting year: 2026
Final Delta table: pr_dap_efs.business_sdpt_abs.nssc_ngs_table_1_2026


year,state,affiliation,lowgrade,highgrade,schooltype,special_ind,primary_size,secondary_size,school_count,enrolments,fte
2026,1,2,1,7,1,Regular,4,1,2,92,92.00
2026,1,2,1,7,1,Regular,5,1,1,112,112.00
2026,1,2,1,7,1,Regular,6,1,2,451,451.00
2026,1,2,1,7,1,Regular,8,1,1,423,423.00
2026,1,2,1,10,3,Regular,10,7,1,934,934.00
2026,1,2,1,11,3,Regular,8,6,1,647,647.00
2026,1,2,1,13,3,Regular,4,5,1,254,254.00
2026,1,2,1,13,3,Regular,5,6,3,1310,1310.00
2026,1,2,1,13,3,Regular,5,7,2,972,972.00
2026,1,2,1,13,3,Regular,5,8,1,638,638.00


School count, enrolment and FTE reconciliation:


source_school_count,source_enrolments,source_fte,output_school_count,output_enrolments,output_fte,school_count_difference,enrolment_difference,fte_difference,status
2953,1573743.000000,1572983.000000,2953,1573743,1572983.00,0,0.000000,0.000000,MATCH


Year-level validation passed: all source year levels mapped to template grade codes.
Completed successfully: NSSC Table 1 matches the template schema and all school count, enrolment and FTE totals reconcile.
